# 🧰 quant-kit — Kaggle Benchmark Suite

**Free benchmarks using Kaggle's T4 GPU.**

- ⚡ Speed (PP/TG) via `llama-cpp-python` CUDA
- 📉 Perplexity (WikiText-2) via `lm-eval`
- 🧠 Downstream benchmarks (TruthfulQA, GPQA, ARC, HellaSwag, GSM8K, Winogrande)
- 📄 Auto-uploads README to HuggingFace

### Setup
1. Add `HF_TOKEN` as a Kaggle Secret
2. Set `HF_REPO` below
3. Enable **GPU T4 x2** → Run All (~5-7 hours)

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
HF_REPO           = "Dhptl/gemma-4-12b-it-GGUF"
ORIGINAL_MODEL_ID = "google/gemma-4-12b-it"
QUANT_TYPE        = "Q4_K_M"

RUN_SPEED = True
RUN_PPL   = True
RUN_EVAL  = True
EVAL_TASKS = "truthfulqa_mc2,gpqa_diamond,arc_challenge,hellaswag,gsm8k,winogrande"
# ──────────────────────────────────────────────────────────────────────

In [ ]:
# ── Install packages ───────────────────────────────────────────────────
import subprocess, sys, os

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-cpp-python",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "lm-eval[api]", "psutil", "datasets", "jinja2"], check=True)

r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print(f"GPU: {r.stdout.strip()}")
print("Packages ready!")

In [ ]:
# ── Setup HF + download main quant ────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
from pathlib import Path
import os, json

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)

model_name = HF_REPO.split("/")[1]
base_name  = model_name.replace("-GGUF", "")

all_files  = list(list_repo_files(HF_REPO, token=HF_TOKEN))
all_quants = sorted([f for f in all_files if f.endswith(".gguf") and "F16" not in f])
print(f"{len(all_quants)} quants found: {[q.split('-')[-1].replace('.gguf','') for q in all_quants]}")

main_gguf_file  = f"{base_name}-{QUANT_TYPE}.gguf"
main_model_path = f"/kaggle/working/{main_gguf_file}"
print(f"Downloading {main_gguf_file}...")
hf_hub_download(repo_id=HF_REPO, filename=main_gguf_file,
                local_dir="/kaggle/working", token=HF_TOKEN)
print(f"Ready: {Path(main_model_path).stat().st_size/1e9:.2f} GB")

In [ ]:
# ── 1. Speed Benchmark (load model ONCE per context, suppress warnings) ─
import time, os, sys, json
from llama_cpp import Llama

speed_results = []

def load_llm_silent(model_path, n_ctx, n_batch):
    """Load Llama model while suppressing C-level stderr warnings."""
    import os
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    old_stderr = os.dup(2)
    os.dup2(devnull_fd, 2)
    os.close(devnull_fd)
    try:
        llm = Llama(
            model_path=model_path,
            n_gpu_layers=-1,
            n_ctx=n_ctx,
            n_batch=n_batch,
            verbose=False,
        )
    finally:
        os.dup2(old_stderr, 2)
        os.close(old_stderr)
    return llm

if RUN_SPEED:
    print("\n" + "="*55)
    print(f"  Speed — {QUANT_TYPE} on T4 GPU")
    print("="*55)

    N_GEN = 128
    REPS  = 2   # 2 reps on the SAME loaded model (fast)

    for ctx in [128, 512, 2048]:
        print(f"  Context {ctx} tokens... ", end="", flush=True)

        # Load model ONCE for this context size
        llm = load_llm_silent(main_model_path, n_ctx=ctx + N_GEN, n_batch=ctx)

        pp_times, tg_times = [], []
        prompt_tokens = llm.tokenize(b"The quick brown fox jumps over the lazy dog. " * 60)[:ctx]

        for rep in range(REPS):
            # Reset KV cache between reps
            llm.reset()

            # ── Prompt Processing (PP) ──
            t0 = time.perf_counter()
            llm.eval(prompt_tokens)
            pp_times.append(len(prompt_tokens) / (time.perf_counter() - t0))

            # ── Token Generation (TG) ──
            t0 = time.perf_counter()
            for _ in range(N_GEN):
                tok = llm.sample()
                llm.eval([tok])
            tg_times.append(N_GEN / (time.perf_counter() - t0))

        del llm  # free VRAM before next context

        pp_avg = round(sum(pp_times) / REPS, 2)
        tg_avg = round(sum(tg_times) / REPS, 2)
        speed_results.append({"context": ctx, "pp_tok_s": pp_avg, "tg_tok_s": tg_avg})
        print(f"TG={tg_avg} tok/s  PP={pp_avg} tok/s")

    print("Speed done!")

In [ ]:
# ── 2. Perplexity via lm-eval wikitext ────────────────────────────────
import subprocess, sys, json
from pathlib import Path

ppl_result = None
ppl_all    = {}

if RUN_PPL:
    print("\n" + "="*55)
    print(f"  Perplexity — {QUANT_TYPE} on WikiText-2")
    print("="*55)

    ppl_dir = Path("/kaggle/working/ppl_results")
    ppl_dir.mkdir(exist_ok=True)

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model", "gguf",
        "--model_args", f"pretrained={main_model_path}",
        "--tasks", "wikitext",
        "--output_path", str(ppl_dir),
        "--device", "cuda",
    ]
    print("Running (~20-30 min)...")
    subprocess.run(cmd, text=True, timeout=7200)

    for result_file in ppl_dir.glob("**/*.json"):
        if "results" in result_file.name:
            with open(result_file) as f:
                data = json.load(f)
            metrics = data.get("results", {}).get("wikitext", {})
            ppl_result = metrics.get("word_perplexity,none") or metrics.get("byte_perplexity,none")
            if ppl_result:
                ppl_result = round(ppl_result, 4)
                ppl_all[QUANT_TYPE] = ppl_result
                print(f"  {QUANT_TYPE} Perplexity = {ppl_result}")
            break
    print("Perplexity done!")

In [ ]:
# ── 3. lm-eval downstream benchmarks ─────────────────────────────────
eval_results = {}

if RUN_EVAL:
    print("\n" + "="*55)
    print(f"  lm-eval — {QUANT_TYPE}: {EVAL_TASKS}")
    print("="*55)

    results_dir = Path("/kaggle/working/eval_results")
    results_dir.mkdir(exist_ok=True)

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model", "gguf",
        "--model_args", f"pretrained={main_model_path}",
        "--tasks", EVAL_TASKS,
        "--output_path", str(results_dir),
        "--batch_size", "auto",
        "--device", "cuda",
    ]
    print("Running (2-4 hours)...")
    subprocess.run(cmd, text=True, timeout=18000)

    for result_file in results_dir.glob("**/*.json"):
        if "results" in result_file.name:
            with open(result_file) as f:
                data = json.load(f)
            for task, metrics in data.get("results", {}).items():
                score = (
                    metrics.get("acc_norm,none") or
                    metrics.get("acc,none") or
                    metrics.get("exact_match,none")
                )
                if score is not None:
                    eval_results[task] = round(score * 100, 2)
                    print(f"  {task}: {eval_results[task]}%")
            break
    print("lm-eval done!")

In [ ]:
# ── 4. Save + upload results JSON ─────────────────────────────────────
output = {
    "model":          HF_REPO,
    "quant":          QUANT_TYPE,
    "platform":       "Kaggle T4 GPU",
    "speed":          speed_results,
    "perplexity_all": ppl_all,
    "perplexity":     ppl_result,
    "benchmarks":     eval_results,
}
result_file = f"/kaggle/working/kaggle_results_{QUANT_TYPE}.json"
with open(result_file, "w") as f:
    json.dump(output, f, indent=2)

api.upload_file(
    path_or_fileobj=result_file,
    path_in_repo=f"kaggle_results_{QUANT_TYPE}.json",
    repo_id=HF_REPO, repo_type="model",
    commit_message=f"Add Kaggle benchmark results ({QUANT_TYPE})"
)
print("Results JSON uploaded!")

In [ ]:
# ── 5. Auto-generate & upload README ──────────────────────────────────
from datetime import datetime
from huggingface_hub import ModelCard

TASK_META = {
    "truthfulqa_mc2": ("TruthfulQA",   "Resistance to hallucination"),
    "gpqa_diamond":   ("GPQA Diamond",  "Hard science reasoning (PhD-level)"),
    "arc_challenge":  ("ARC Challenge", "Grade-school science reasoning"),
    "hellaswag":      ("HellaSwag",     "Common sense completion"),
    "gsm8k":          ("GSM8K",         "Grade-school math word problems"),
    "winogrande":     ("Winogrande",    "Commonsense pronoun resolution"),
}
try:
    license_ = ModelCard.load(ORIGINAL_MODEL_ID).data.get("license", "other")
except Exception:
    license_ = "other"

ppl_row   = f"| `{QUANT_TYPE}` | `{ppl_result}` |\n" if ppl_result else ""
ppl_table = f"| Quant | Perplexity (WikiText-2) ↓ |\n|---|---|\n{ppl_row}"

bench_table = "| Benchmark | Score | What it measures |\n|---|---|---|\n"
for task, score in eval_results.items():
    name, desc = TASK_META.get(task, (task, ""))
    bench_table += f"| **{name}** | `{score}%` | {desc} |\n"

speed_table = "| Context | Token Generation | Prompt Processing |\n|---|---|---|\n"
for r in speed_results:
    tg = f"{r['tg_tok_s']} tok/s" if r.get('tg_tok_s') else "—"
    pp = f"{r['pp_tok_s']} tok/s" if r.get('pp_tok_s') else "—"
    speed_table += f"| {r['context']} tokens | {tg} | {pp} |\n"

readme = f"""---
license: {license_}
base_model: {ORIGINAL_MODEL_ID}
pipeline_tag: text-generation
tags:\n  - gguf\n  - quantized\n  - text-generation
language:\n  - en
---

<div align=\"center\">

# {model_name}

Quantized GGUF versions of [{ORIGINAL_MODEL_ID}](https://huggingface.co/{ORIGINAL_MODEL_ID}).  
Works with llama.cpp, Ollama, LM Studio, and any GGUF-compatible runtime.

*Benchmarked on Kaggle T4 GPU · {datetime.now().strftime("%B %d, %Y")} · Built with [quant-kit](https://github.com/DhruvalPtl/quant-kit)*

</div>

---

## ⚖️ Quality — Perplexity on WikiText-2

Lower = closer to original FP16 quality.

{ppl_table}

---

## 🧠 Downstream Benchmarks — `{QUANT_TYPE}` on T4 GPU

*Evaluated using [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)*

{bench_table}

---

## ⚡ Speed — `{QUANT_TYPE}` on Kaggle T4 GPU

{speed_table}

---

## 🚀 How to Use

### llama.cpp
```bash
./llama-cli -m {base_name}-Q4_K_M.gguf -p \"Your prompt\" -n 512
```

### Python
```python
from llama_cpp import Llama
llm = Llama(model_path=\"./{base_name}-Q4_K_M.gguf\", n_gpu_layers=-1)
print(llm(\"Tell me about AI\", max_tokens=256)[\"choices\"][0][\"text\"])
```

---
*Quantized with [quant-kit](https://github.com/DhruvalPtl/quant-kit)*
"""

with open("/kaggle/working/README.md", "w") as f:
    f.write(readme)
api.upload_file(
    path_or_fileobj="/kaggle/working/README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO, repo_type="model",
    commit_message="Auto-update README with Kaggle benchmark results"
)
print(f"\n✅ README uploaded → https://huggingface.co/{HF_REPO}")
print("🎉 All done!")
print(json.dumps(output, indent=2))